# IndexCalc — Phase 5 테스트

LaTeX 출력 & Jupyter 렌더링. 셀 실행 결과가 LaTeX로 렌더링되는지 확인.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from indexcalc import (
    IndexSpace, Tensor, IndexRegistry, parse, to_latex,
    MetricRegistry, raise_index, lower_index, absorb_metric,
    trace,
)

In [2]:
# Setup
spacetime = IndexSpace("spacetime", dim=4, indices="μνλρσ", metric="g")
lorentz   = IndexSpace("lorentz",   dim=4, indices="abcde", metric="η")

reg = IndexRegistry()
reg.register(spacetime)
reg.register(lorentz)

g     = Tensor("g", [spacetime.lower("μ"), spacetime.lower("ν")])
g_inv = Tensor("g", [spacetime.upper("μ"), spacetime.upper("ν")])
eta     = Tensor("η", [lorentz.lower("a"), lorentz.lower("b")])
eta_inv = Tensor("η", [lorentz.upper("a"), lorentz.upper("b")])

metrics = MetricRegistry()
metrics.register(g, g_inv, spacetime)
metrics.register(eta, eta_inv, lorentz)

## 1. 단일 텐서 — Jupyter 렌더링

셀의 마지막 표현식이 자동으로 LaTeX 렌더링됩니다.

In [3]:
parse("T^{μ}_{ν}", reg)

T^μ_ν

In [4]:
parse("g_{μν}", reg)

g_μ_ν

In [5]:
parse("R^{μ}_{νλρ}", reg)

R^μ_ν_λ_ρ

## 2. 텐서곱 & contraction

In [6]:
parse("T^{μ}_{ν} S^{ν}_{λ}", reg)

(T^μ_ν * S^ν_λ)  [contracted: ν]

In [7]:
parse("g_{μν} V^{ν}", reg)

(g_μ_ν * V^ν)  [contracted: ν]

In [8]:
parse("T^{μ}_{ν} g^{νλ} S_{λρ}", reg)

((T^μ_ν * g^ν^λ)  [contracted: ν] * S_λ_ρ)  [contracted: λ]

## 3. 스칼라곱 & 분수

In [9]:
parse(r"\frac{1}{2} T^{μ}_{ν}", reg)

0.5 * T^μ_ν

In [10]:
parse(r"\frac{3}{4} g_{μν}", reg)

0.75 * g_μ_ν

## 4. 합/차

In [11]:
parse("A^{μ}_{ν} + B^{μ}_{ν}", reg)

(A^μ_ν + B^μ_ν)

In [12]:
parse("A^{μ}_{ν} - B^{μ}_{ν}", reg)

(A^μ_ν + (-B^μ_ν))

In [13]:
parse(r"\frac{1}{2} A^{μ}_{ν} - \frac{1}{3} B^{μ}_{ν}", reg)

(0.5 * A^μ_ν + -0.3333333333333333 * B^μ_ν)

## 5. 괄호 + 곱

In [14]:
parse("(A^{μ}_{ν} + B^{μ}_{ν}) V^{ν}", reg)

((A^μ_ν + B^μ_ν) * V^ν)  [contracted: ν]

## 6. Metric raise/lower → LaTeX

In [15]:
V = parse("V_{μ}", reg)
raise_index(V, "μ", metrics)

(g^μ^μ_1 * V_μ_1)  [contracted: μ_1]

In [16]:
absorb_metric(parse("g_{μν} V^{ν}", reg), metrics)

V_μ

## 7. Trace

In [17]:
T = Tensor("T", [spacetime.upper("μ"), spacetime.lower("μ")])
trace(T, "μ")

Tr(T^μ_μ)

In [18]:
# Partial trace: R^{μa}_{μb} → free indices a, b
R = Tensor("R", [
    spacetime.upper("μ"),
    lorentz.upper("a"),
    spacetime.lower("μ"),
    lorentz.lower("b"),
])
trace(R, "μ")

Tr_μ(R^μ^a_μ_b)

## 8. to_latex() 직접 호출 & 비교

In [19]:
expr = parse("T^{μ}_{ν} g^{νλ} S_{λρ}", reg)
print(f"repr:       {expr}")
print(f"to_latex(): {to_latex(expr)}")
print(f"_repr_latex_(): {expr._repr_latex_()}")

repr:       ((T^μ_ν * g^ν^λ)  [contracted: ν] * S_λ_ρ)  [contracted: λ]
to_latex(): T^{\mu}{}_{\nu} g^{\nu \lambda} S_{\lambda \rho}
_repr_latex_(): $T^{\mu}{}_{\nu} g^{\nu \lambda} S_{\lambda \rho}$


## 9. Vielbein 예시

In [20]:
parse("e^{a}_{μ} e^{b}^{μ} η_{ab}", reg)

((e^a_μ * e^b^μ)  [contracted: μ] * η_a_b)  [contracted: a, b]

## 10. absorb / raise / lower 활용 예시

metric 연산의 결과가 LaTeX로 렌더링되는 걸 확인합니다.

In [21]:
# g_{μν} V^{ν} → absorb → V_{μ}
from IPython.display import display, Math

expr = parse("g_{μν} V^{ν}", reg)
result = absorb_metric(expr, metrics)

display(Math(to_latex(expr) + r" \;\;\xrightarrow{\text{absorb}}\;\; " + to_latex(result)))

<IPython.core.display.Math object>

In [22]:
# g^{μν} T_{νλ} → absorb → T^{μ}_{λ}
expr = parse("g^{μν} T_{νλ}", reg)
result = absorb_metric(expr, metrics)

display(Math(to_latex(expr) + r" \;\;\xrightarrow{\text{absorb}}\;\; " + to_latex(result)))

<IPython.core.display.Math object>

In [23]:
# V_{μ} → raise → g^{μ,μ_1} V_{μ_1}
V = parse("V_{μ}", reg)
raised = raise_index(V, "μ", metrics)

display(Math(to_latex(V) + r" \;\;\xrightarrow{\text{raise}}\;\; " + to_latex(raised)))

<IPython.core.display.Math object>

In [24]:
# 왕복: V_{μ} → raise → absorb → V^{μ}
V = parse("V_{μ}", reg)
step1 = raise_index(V, "μ", metrics)
step2 = absorb_metric(step1, metrics)

display(Math(
    to_latex(V)
    + r" \;\;\xrightarrow{\text{raise}}\;\; "
    + to_latex(step1)
    + r" \;\;\xrightarrow{\text{absorb}}\;\; "
    + to_latex(step2)
))

<IPython.core.display.Math object>

In [25]:
# Lorentz metric absorb: η_{ab} V^{b} → V_{a}
expr = parse("η_{ab} V^{b}", reg)
result = absorb_metric(expr, metrics)

display(Math(to_latex(expr) + r" \;\;\xrightarrow{\text{absorb}}\;\; " + to_latex(result)))

<IPython.core.display.Math object>